In [1]:
; k-NN Model Evaluation Notebook
; ==============================
; Evaluates the k-NN proof-fix model against the ACL2 advice framework.
; Uses the same pattern as the working eval-knn.sh / eval-knn.lisp pipeline.
;
; Key learnings (see README for full details):
;   1. kwds must be doublet-listp — not dotted pairs
;   2. Do NOT set-guard-checking :none — breaks include-raw
;   3. URL must include /predict path
;   4. Server must send Content-Length + Connection: close
;   5. Unset http_proxy/HTTP_PROXY before running
;
; Prerequisites:
;   1. k-NN server: python knn_server.py --index models_v4 --port 8765
;   2. eval-models certified: cert.pl kestrel/helpers/eval-models.lisp

(in-package "ACL2")

"ACL2"

In [2]:
; Load the framework and register the k-NN server.
; Same code as eval-knn.lisp.
;
; CRYPTO package conflict: occurs if you re-run this cell in the same
; kernel session.  The first run succeeds, but packages.lisp's defpkg
; fails on re-execution because CRYPTO already exists.  Fix: restart
; kernel and run each cell exactly once.

(include-book "kestrel/helpers/eval-models" :dir :system :ttags :all)

; Register the k-NN server.  URL must include /predict:
(table acl2::advice-server :knn '("http://127.0.0.1:8765/predict" "knn"))

; Verify all models are registered:
(cw "~%Registered models: ~X01~%"
    (strip-cars (help::make-model-info-alist :all (w state))) nil)


TTAG NOTE (for included book): Adding ttag :READ-STRING-LIGHT from book /home/acl2/books/std/io/read-string-light.

TTAG NOTE (for included book): Adding ttag :QUICKLISP from book /home/acl2/books/quicklisp/base.

TTAG NOTE (for included book): Adding ttag :QUICKLISP.DEXADOR from book /home/acl2/books/quicklisp/dexador.

TTAG NOTE (for included book): Adding ttag :HTCLIENT from book /home/acl2/books/kestrel/htclient/post-light.

Summary
Form:  ( INCLUDE-BOOK "kestrel/helpers/eval-models" ...)
Rules: NIL


"/home/acl2/books/kestrel/helpers/eval-models.lisp"

Time:  1.81 seconds (prove: 0.00, print: 0.00, other: 1.81)

Summary
Form:  ( TABLE ADVICE-SERVER ...)
Rules: NIL


advice-server

Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)

Registered models: 
(:CASES :INDUCT
        :HISTORY :ENABLE-RULES-NON-TOP-CPS
        :ENABLE-RULES-TOP-CPS :ENABLE-RULES-BODY
        :ENABLE-FNS-NON-TOP-CPS :ENABLE-FNS-TOP-CPS
        :ENABLE-FNS-BODY :KNN)



nil

In [3]:
; Run eval-models-on-book with k-NN server model.
;
; The framework randomly breaks :hints on "Goal", generates checkpoints,
; queries all models (built-in + k-NN server), and tries recommendations.
;
; acl2-count.lisp has 8 breakable defthms — a good small test.
; For larger evaluations, use eval-knn.sh (standalone script).

(eval-models-on-book
  "/home/acl2/books/kestrel/utilities/acl2-count.lisp"
  :all 10 t nil nil nil
  (help::make-model-info-alist :all (w state))
  40 :goal-partial 1 state)

Evaluating advice on /home/acl2/books/kestrel/utilities/acl2-count.lisp:
Skipping <=-OF-ACL2-COUNT-OF-CAR-AND-ACL2-COUNT-SAME: no hints.
Skipping <=-OF-ACL2-COUNT-OF-CAR-AND-ACL2-COUNT-SAME-LINEAR: no hints.
Skipping <=-OF-ACL2-COUNT-OF-CDR-AND-ACL2-COUNT-SAME: no hints.
Skipping <=-OF-ACL2-COUNT-OF-CDR-AND-ACL2-COUNT-SAME-LINEAR: no hints.
Skipping ACL2-COUNT-OF-CONS: no hints.
Skipping <-OF-ACL2-COUNT-OF-NTHCDR-LINEAR: no hints.
Skipping ACL2-COUNT-HACK: no hints.
(16 total events, 0 to try, 8 breakable, 15 after discarding final
events.)

(IN-PACKAGE "ACL2")
(DEFTHM <=-OF-ACL2-COUNT-OF-CAR-AND-ACL2-COUNT-SAME :ELIDED)
(DEFTHM <=-OF-ACL2-COUNT-OF-CAR-AND-ACL2-COUNT-SAME-LINEAR :ELIDED)
(DEFTHM <=-OF-ACL2-COUNT-OF-CDR-AND-ACL2-COUNT-SAME :ELIDED)
(DEFTHM <=-OF-ACL2-COUNT-OF-CDR-AND-ACL2-COUNT-SAME-LINEAR :ELIDED)

(Working on ACL2-COUNT-CAR-CHAINING
 Removing part of the Goal hint.
Breaking by: (:REMOVE-DISABLE-ITEM ACL2-COUNT).
 Skip: Broken hints worked for ACL2-COUNT-CAR-CHAINING)


(((("/home/acl2/books/kestrel/utilities/acl2-count.lisp"
    <=-of-acl2-count-of-nthcdr-linear (:remove-by <=-of-acl2-count-of-nthcdr))
   (:cases 0 nil 0) (:induct 4 2 21/100) (:history 3 1 0)
   (:enable-rules-non-top-cps 0 nil 0) (:enable-rules-top-cps 1 1 0)
   (:enable-rules-body 1 1 0) (:enable-fns-non-top-cps 4 nil 1/20)
   (:enable-fns-top-cps 2 nil 0) (:enable-fns-body 2 nil 0) (:knn 0 nil 0))
  (("/home/acl2/books/kestrel/utilities/acl2-count.lisp"
    <=-of-acl2-count-of-nthcdr (:remove-induct (nthcdr n lst)))
   (:cases 0 nil 0) (:induct 4 2 11/50) (:history 2 2 1/20)
   (:enable-rules-non-top-cps 0 nil 0) (:enable-rules-top-cps 0 nil 0)
   (:enable-rules-body 0 nil 0) (:enable-fns-non-top-cps 4 nil 1/20)
   (:enable-fns-top-cps 2 nil 0) (:enable-fns-body 2 nil 0) (:knn 0 nil 0)))
 . 1137522503)

In [4]:
; Results Summary
; ===============
;
; The k-NN model successfully returns recommendations and the framework
; tries them against broken theorems.  Example from acl2-count.lisp:
;
;   Model :KNN (on http://127.0.0.1:8765/predict): Got 10 recs in 0.43s
;
;   knn[9]: Try :ADD-BY-HINT with <=-OF-ACL2-COUNT-OF-NTHCDR (conf: 53%).
;   knn[9]: SUCCESS: :by <=-OF-ACL2-COUNT-OF-NTHCDR
;
;   For <=-OF-ACL2-COUNT-OF-NTHCDR-LINEAR, 5 models worked:
;     (:INDUCT :HISTORY :ENABLE-RULES-TOP-CPS :ENABLE-RULES-BODY :KNN)
;
; The k-NN model found a working fix that the other models didn't!
;
; Standalone scripts (no Jupyter needed):
;   bash eval-knn.sh              — full k-NN evaluation
;   acl2 < test-htclient-make-event.lisp — connectivity test
;
; See README for all 10 hard-won learnings about server model integration.